# IAD Pipeline — Training
Anomaly detection pipeline using FiftyOne, Weights & Biases, and the IAD framework.

This notebook uses the refactored `AnomalyDetectionManager`:
- `train()` / `eval()` no longer take a `tiling` flag or call `adjustPaths()` manually — tiled-ensemble is currently the only supported mode, and paths are resolved internally by `_prepareRun`.
- Before calling an action, we check `manager.can_run(...)` / `manager.get_missing_requirements(...)` so we can show the user *why* something isn't ready yet instead of hitting a bare exception.
- State-related failures raise `ManagerStateError` subclasses (`NoModelLoadedError`, `NoDatasetLoadedError`, `TilingNotConfiguredError`, `ModelNotTrainedError`, `CheckpointNotFoundError`), each carrying a `.missing` list.

## 1. Environment Setup
Configure database URI and API keys.

In [1]:
import os
import sys
import warnings

# Set BEFORE any fiftyone imports
os.environ["FIFTYONE_DATABASE_URI"] = "mongodb://localhost"
os.environ["WANDB_API_KEY"] = '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'

sys.path.append("src")

import wandb
import logging
from pathlib import Path

from src.manager import AnomalyDetectionManager as ADM
from src.manager import DatasetSession as DS
from src.manager import (
    # ManagerState,
    ManagerError,
    ManagerStateError,
    # ConfigError,
    # NoModelLoadedError,
    # NoDatasetLoadedError,
    # TilingNotConfiguredError,
    # ModelNotTrainedError,
    CheckpointNotFoundError,
)
from src.tiling.tilingCheckpoints import checkTiledCheckpointsExist


warnings.filterwarnings("ignore", category=FutureWarning, module="timm.models.layers")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="openvino.runtime")

logger = logging.getLogger("logger")

wandb.login()

/Users/dapo/Documents/Code/IAD/.venv/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
wandb: Currently logged in as: daniel-pommer (daniel-pommer-technische-hochschule-n-rnberg-georg-simon-ohm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 2. Configuration
Set your run parameters here before executing the pipeline.

`ADM.loadProduct(...)` loads the model, configures tiling, and resolves output/checkpoint
paths for you — no manual `adjustPaths()` call needed afterwards.

In [2]:
# from src.userConfigs import Product
from src.setup import Product

datasetDir  = Path("datasets/")
configDir   = Path("configs/")
outputPath  = Path("results/")
productPath = Path("Products/cable.yaml")
productConfigPath = Path(configDir / productPath)

product: Product
manager, product = ADM.loadProduct(
    productConfigPath=productConfigPath,
    outputPath=outputPath,
    configDir=configDir,
)

print(f"Loaded product: {product.name}")
print(product)
print(f"Manager state: {manager.state!r}")

INFO: Initializing InpFormer model.


/Users/dapo/Documents/Code/IAD/.venv/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'pre_processor' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['pre_processor'])`.
/Users/dapo/Documents/Code/IAD/.venv/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'post_processor' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['post_processor'])`.
/Users/dapo/Documents/Code/IAD/.venv/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'evaluator' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['evaluator'])`.


INFO: Loaded model: dinov2reg_vit_base_14
INFO: Successfully loaded model InpFormer: InpFormer(
  (pre_processor): PreProcessor(
    (transform): Compose(    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], inplace=False))
    (export_transform): Compose(    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], inplace=False))
  )
  (post_processor): AOIPostProcessor(
    (_image_threshold_metric): F1AdaptiveThreshold()
    (_pixel_threshold_metric): F1AdaptiveThreshold()
    (_image_min_max_metric): MinMax()
    (_pixel_min_max_metric): MinMax()
  )
  (evaluator): Evaluator(
    (val_metrics): ModuleList(
      (0): AUROC()
    )
    (test_metrics): ModuleList(
      (0): AUROC()
      (1): F1Score()
      (2): AUPR()
    )
  )
  (model): InpFormerModel(
    (encoder): DinoVisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
        (norm): Identity()
      )
      (blocks): ModuleList(
   

## 3. Load & Select Dataset

We load the dataset session and select the product's category. `DatasetSession` is
independent of the manager's readiness state — you can load and inspect data before
a model is ready, or vice versa.

In [3]:
datasetSession = DS.loadDatasetFromConfig(product.datasetConfig, overwrite=False, merge=False)
datasetSession.select_category(product.name)

print(f"Dataset: {datasetSession.datasetName}, category: {datasetSession.category}")
print(f"Images in view: {len(datasetSession.FO_Dataset)}")

INFO: Dataset 'MVTecADShort' already exists in database
INFO: Loading from database
INFO: Loaded dataset 'MVTecADShort' from database!
INFO: Selected category: cable, 374 images


ERROR: Dataset name 'MVTecADShort-cable' is not available


ERROR: Dataset name 'MVTecADShort-cable' is not available
INFO: Deleting MVTecADShort-cable from database and reloading.
Dataset: MVTecADShort, category: cable
Images in view: 374


## 3.1 Inspect Dataset

In [ ]:
datasetSession.launchSession()

## 4. Readiness Check

Before training, ask the manager what (if anything) is missing, rather than firing the
call and translating an exception. This is the check a UI would run to enable/disable
a "Train" button.

In [4]:
manager.attachDatasetSession(datasetSession)
missing = manager.get_missing_requirements("train")
if missing:
    print("Not ready to train yet. Missing:")
    for item in missing:
        print(f"  - {item}")
else:
    print("Ready to train.")
# manager._prepareRun(trainerConfig=product.trainerConfig,
#     modelConfig=product.modelConfig,
#     datamoduleConfig=product.datamoduleConfig,
#     datasetSession=datasetSession,
#     tilingPipelineConfig=product.tilingPipelineConfig,)

INFO: Attached dataset 'MVTecADShort' (category=cable)
Ready to train.


## 5. Training

`train()` is tiled-ensemble only for now (the `tiling` parameter has been removed —
non-tiled support will be added later as an explicit branch, once implemented). Paths,
tiling setup, callbacks, and the W&B logger are all handled internally by `_prepareRun`.

We wrap the call so that a `ManagerStateError` (e.g. someone re-running this cell before
`generateModel`/`setupTiling` ran) produces a clear message instead of a raw traceback.

In [ ]:
try:

    manager.train(
        trainerConfig=product.trainerConfig,
        modelConfig=product.modelConfig,
        datamoduleConfig=product.datamoduleConfig,
        datasetSession=datasetSession,
        tilingPipelineConfig=product.tilingPipelineConfig,
    )
    print("Training complete.")
except ManagerStateError as e:
    print(f"Could not train: {e}")
    print(f"Missing: {e.missing}")
except ManagerError as e:
    print(f"Training failed: {e}")

print(f"Manager state: {manager.state!r}")

In [ ]:
datasetSession.launchSession()
# datasetSession.save()

## 5.1 Training Loss Visualization

Each tile in the ensemble trains a separate model and logs to its own W&B run
(`AnomalibWandbLogger`, configured in `configs/Trainer/Training_InpFormer.yaml` via the
`logger:` key). The engine gives each tile a deterministic run id, grouped under this
ensemble run, and writes that id once to `manager.wandbManifestDir/model<i>_<j>.json` — we
just read those files back rather than re-deriving the id (see
`AOITiledEnsembleEngine._setup_anomalib_callbacks` in `src/tiling/ensemble_engine.py`, and
`src/run_paths.py` for the shared path/naming conventions manager.py and the engine both
draw from).

We pull each tile's full (unsampled) `train_loss_step` history via `wandb.Api()`, save it
raw to CSV next to the run, and plot it raw — no smoothing. Smoothing is better done
on-demand, either on the W&B dashboard (EMA slider) or from the saved CSVs.

This only works for training runs made **after** the W&B logger was wired up — older runs
under `results/` have no `wandb_runs/` directory.

In [ ]:
import json

import pandas as pd
import matplotlib.pyplot as plt

runDir = Path(manager.outputDir)
manifestDir = manager.wandbManifestDir  # single source of truth, see src/run_paths.py
plotsDir = runDir / "plots"

tileHistories = {}
if manifestDir.exists():
    plotsDir.mkdir(parents=True, exist_ok=True)
    api = wandb.Api()

    for manifestFile in sorted(manifestDir.glob("*.json")):
        entry = json.loads(manifestFile.read_text())
        tileName = manifestFile.stem
        runPath = f"{entry['entity']}/{entry['project']}/{entry['id']}"

        try:
            run = api.run(runPath)
        except Exception as e:
            print(f"Could not fetch {runPath}: {e}")
            continue

        # scan_history (not history()) returns every logged row, unsampled - history()
        # silently downsamples to ~500 points, which would smooth the curve implicitly.
        rows = list(run.scan_history(keys=["trainer/global_step", "train_loss_step"]))
        history = pd.DataFrame(rows).dropna(subset=["train_loss_step"])
        if history.empty:
            continue

        tileHistories[tileName] = history
        history.to_csv(plotsDir / f"train_loss_{tileName}.csv", index=False)
else:
    print(f"No W&B run manifest found under {manifestDir}.")
    print("This training run predates the W&B logger wiring, or hasn't been run yet.")

if not tileHistories:
    print("No W&B training history could be retrieved.")
else:
    fig, ax = plt.subplots(figsize=(9, 5))
    colors = plt.get_cmap("tab10")

    for i, (tileName, history) in enumerate(tileHistories.items()):
        ax.plot(
            history["trainer/global_step"], history["train_loss_step"],
            label=tileName, color=colors(i % 10), linewidth=2,
        )

    ax.set_title("Train loss per tile (raw)")
    ax.set_xlabel("Step")
    ax.set_ylabel("train_loss")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()

    plotPath = plotsDir / "train_loss.png"
    fig.savefig(plotPath, dpi=150)
    print(f"Saved raw training loss plot to {plotPath}")
    plt.show()

## 6. Evaluation

`eval()` requires the model to actually be trained (`ManagerState.TRAINED`), which is
only set once `train()` confirms a checkpoint landed on disk — not just that the call
returned. Check readiness first, same pattern as training.

In [ ]:
# ckptDir = Path("results/MVTecADShort/cable/Padim/tiled/checkpoints")
# ckptDir = manager.ckptDir
# if ckptDir is None:
#     raise ValueError
print(manager.modelTrainingDir)
print(manager.ckptDir)

if manager.modelTrainingDir is None and manager.ckptDir is not None:
    manager.modelTrainingDir = manager.ckptDir.parent
missing = manager.get_missing_requirements("eval")
print(missing)
# manager.loadCheckpoint(ckptDir, product.tilingPipelineConfig)

# if missing:
#     print("Not ready to evaluate yet. Missing:")
#     for item in missing:
#         print(f"  - {item}")

# if ManagerState.CHECKPOINT_AVAILABLE:
try:
    manager.eval(
        evalConfig=product.trainerConfig,
        modelConfig=product.modelConfig,
        datamoduleConfig=product.datamoduleConfig,
        datasetSession=datasetSession,
        tilingPipelineConfig=product.tilingPipelineConfig,
    )
    print("Evaluation complete.")
except ManagerStateError as e:
    print(f"Could not evaluate: {e}")
except ManagerError as e:
    print(f"Evaluation failed: {e}")

In [ ]:
datasetSession.launchSession()

## 6.1 Anomaly Score Distribution & Threshold

`eval()` writes the post-processing statistics (image/pixel threshold, min/max used for
normalization) to `stats.json` in the run directory, and writes each sample's raw
anomaly score into the FiftyOne dataset as `pred_anomaly_score_<model_name>` (see
`AOIStatisticsJob` / `AOIFiftyOneVisJob` in `src/tiling/jobs.py`). Neither is kept on
`manager` in memory, so we read both back here: the threshold from disk, the per-image
scores from the dataset.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

modelName = product.modelConfig.name
scoreField = f"pred_anomaly_score_{modelName}"

runDir = Path(manager.modelTrainingDir) if manager.modelTrainingDir is not None else Path(manager.outputDir)
statsCandidates = [runDir / "stats.json", runDir / "checkpoints" / "stats.json"]
statsPath = next((p for p in statsCandidates if p.exists()), None)

if statsPath is None:
    print(f"No stats.json found under {runDir}. Checked: {[str(p) for p in statsCandidates]}")
    print("Run eval() first.")
else:
    with statsPath.open() as f:
        stats = json.load(f)
    imageThreshold = stats["image_threshold"]
    print(f"Loaded stats from {statsPath}")
    print(f"Image threshold: {imageThreshold:.4f}")

    predictedView = datasetSession.FO_Dataset.match_tags("predicted")
    scores = [s[scoreField] for s in predictedView.select_fields(scoreField) if s[scoreField] is not None]

    if not scores:
        print(f"No samples with field '{scoreField}' found - run eval() to populate anomaly scores.")
    else:
        scores = np.array(scores)
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.hist(scores, bins=30, color="#4C72B0", alpha=0.85, edgecolor="white", linewidth=0.5)
        ax.axvline(
            imageThreshold, color="#C44E52", linewidth=2, linestyle="--",
            label=f"Threshold = {imageThreshold:.3f}",
        )
        ax.set_xlabel("Anomaly score")
        ax.set_ylabel("Number of images")
        ax.set_title(f"Anomaly score distribution — {modelName}")
        ax.set_xlim(0.0,1.0)
        ax.grid(alpha=0.3)
        ax.legend()
        fig.tight_layout()
        plt.show()

## 7. Evaluate on Unknown / Prediction Data

Inference reads its checkpoint from an explicit `trainingDir` (which may belong to a
different manager/session than the one currently in memory) rather than relying on
`self.ckptDir` from the last training run. `inference()` checks that the checkpoint
file actually exists at that path and raises `CheckpointNotFoundError` if not — state
flags alone can't guarantee this, since `trainingDir` is caller-supplied.

This cell is left commented out, matching the placeholder in the original notebook — 
uncomment and adjust `predDatasetName` / `predDatasetDir` to run a prediction pass.

In [ ]:
predDatasetName = "MVTecADShortPred"
predDatasetDir = datasetDir / predDatasetName

from src.setup import DataModuleConfig

predSession = DS.loadDatasetFromDisk(
    predDatasetDir,
    datasetName=predDatasetName,
    overwrite=False,
    merge=False,
    split=("pred",),
)
predSession.select_category(product.name)

manager.modelTrainingDir = product.refresh_training_dir(manager.baseOutputDir)

missing = manager.get_missing_requirements("inference")
if missing:
    print("Not ready for inference yet. Missing:")
    for item in missing:
        print(f"  - {item}")
else:
    try:
        manager.inference(
            inferencerConfig=product.inferencerConfig,
            modelConfig=product.modelConfig,
            modelTrainingDir=product.modelTrainingDir,
            datamoduleConfig=DataModuleConfig.load_datamodule_config_from_yaml(product.inferencerConfigPath),
            datasetSession=predSession,
            tilingPipelineConfig=product.tilingPipelineConfig,
        )
        print("Inference complete.")
    except CheckpointNotFoundError as e:
        print(f"No checkpoint available: {e}")
    except ManagerStateError as e:
        print(f"Could not run inference: {e}")
    except ManagerError as e:
        print(f"Inference failed: {e}")

predSession.launchSession()

ERROR: Dataset name 'MVTecADShortPred' is not available


ERROR: Dataset name 'MVTecADShortPred' is not available
INFO: There are 20 images in the MVTecADShortPred dataset.
INFO: There are 1 categorie(s) in the MVTecADShortPred dataset.
INFO: ['cable']
INFO: Selected category: cable, 20 images


ERROR: Dataset name 'MVTecADShortPred-cable' is not available


ERROR: Dataset name 'MVTecADShortPred-cable' is not available
INFO: Deleting MVTecADShortPred-cable from database and reloading.
INFO: Attached dataset 'MVTecADShortPred' (category=cable)
{'trainer': {'accelerator': 'mps', 'strategy': 'auto', 'devices': 'auto', 'num_nodes': 1, 'fast_dev_run': False, 'max_epochs': 1, 'max_steps': -1, 'overfit_batches': 0.0, 'check_val_every_n_epoch': 1, 'accumulate_grad_batches': 1, 'inference_mode': True, 'use_distributed_sampler': True, 'profiler': 'simple', 'detect_anomaly': False, 'barebones': False, 'sync_batchnorm': False, 'reload_dataloaders_every_n_epochs': 0}, 'model': {'name': 'InpFormer', 'preProcessorPath': '/Users/dapo/Documents/Code/IAD/configs/Engine/PreProcessor.yaml', 'postProcessorPath': '/Users/dapo/Documents/Code/IAD/configs/Engine/PostProcessor.yaml', 'evaluatorPath': '/Users/dapo/Documents/Code/IAD/configs/Engine/Evaluator.yaml'}, 'datamodule': {'val_split_mode': 'from_test', 'test_split_mode': 'from_dir', 'train_batch_size': 'auto

            might be empty or devoid of either normal or anomalous images.


            might be empty or devoid of either normal or anomalous images.


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:22                                                                                   │
│                                                                                                  │
│   19 │   │   print(f"  - {item}")                                                                │
│   20 else:                                                                                       │
│   21 │   try:                                                                                    │
│ ❱ 22 │   │   manager.inference(                                                                  │
│   23 │   │   │   inferencerConfig=product.inferencerConfig,                                      │
│   24 │   │   │   modelConfig=product.modelConfig,                                                │
│   25 │   │   │   modelTrainingDir=product.modelTrainingDir,                                      │
│                                                                                                  │
│ /Users/dapo/Documents/Code/IAD/src/manager.py:883 in inference                                   │
│                                                                                                  │
│   880 │   │   # self._apply_visualizer_output_dir(ctx.outputDir)                                 │
│   881 │   │   # self.setupTiling(tilingPipelineConfig)                                           │
│   882 │   │                                                                                      │
│ ❱ 883 │   │   self._inferenceTiledModel(                                                         │
│   884 │   │   │   trainingDir=modelTrainingDir,                                                  │
│   885 │   │   │   ckptDir=self.ckptDir,                                                          │
│   886 │   │   │   datasetSession=datasetSession,                                                 │
│                                                                                                  │
│ /Users/dapo/Documents/Code/IAD/src/manager.py:721 in _inferenceTiledModel                        │
│                                                                                                  │
│   718 │   │   │   How the tiling process works (E.g. tile size, stride,...)                      │
│   719 │   │   """                                                                                │
│   720 │   │                                                                                      │
│ ❱ 721 │   │   datamodule = datasetSession.setupDatamodule(dataModuleConfig, self.outputDir)      │
│   722 │   │                                                                                      │
│   723 │   │   logger.info(f"Dataset used for training: {datasetSession.FO_Dataset}")             │
│   724                                                                                            │
│                                                                                                  │
│ /Users/dapo/Documents/Code/IAD/src/setup.py:895 in setupDatamodule                               │
│                                                                                                  │
│    892 │   │   │   if dm_kwargs.get(key) == "auto":                                              │
│    893 │   │   │   │   dm_kwargs[key] = 2                                                        │
│    894 │   │   datamodule = FODataModule(name=self.datasetName, samples=self.FO_Dataset, root=o  │
│ ❱  895 │   │   datamodule.setup()                                                                │
│    896 │   │   self.datamodule = datamodule                                                      │
│    897 │   │   return self.datamodule                                                            │
│    898                                                     